In [1]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as mlines
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import MultipleLocator, MaxNLocator
import math
import io
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ==============================================================================
# --- STYLE CONFIGURATION & INJECTION ---
# ==============================================================================
css_code = """
<style>
/* Memaksa semua tulisan widget (label, input, dropdown) menjadi HITAM NORMAL (Clean) */
.widget-label, .widget-readout, .widget-text input, .widget-numeric-text input, .widget-dropdown select {
    color: #000000 !important;
    font-weight: 600 !important;
}
.jupyter-widgets {
    color: #000000 !important;
}

/* Mengamankan layout agar tidak memunculkan scrollbar (auto-expand) */
.output_wrapper, .output, .output_scroll, .widget-output, .jp-OutputArea-child {
    height: auto !important; max-height: none !important; overflow: hidden !important;
    box-shadow: none !important; -webkit-box-shadow: none !important;
}
.widget-hbox { flex-wrap: wrap !important; gap: 5px; }
</style>
"""
display(HTML(css_code))

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['text.color'] = '#1a1a1a'
plt.rcParams['axes.labelcolor'] = '#1a1a1a'
plt.rcParams['xtick.color'] = '#1a1a1a'
plt.rcParams['ytick.color'] = '#1a1a1a'

# ==============================================================================
# --- 1. DATABASE SPESIFIKASI ---
# ==============================================================================
SPUN_PILE_DATA = {
    300: {'t': 60, 'wires': {'A1':(6, 7.1), 'A2':(6, 7.1), 'A3':(8, 7.1), 'B':(8, 7.1), 'C':(10, 7.1)}},
    350: {'t': 65, 'wires': {'A1':(6, 7.1), 'A2':(8, 7.1), 'A3':(10, 7.1), 'B':(10, 7.1), 'C':(12, 7.1)}},
    400: {'t': 75, 'wires': {'A1':(8, 7.1), 'A2':(10, 7.1), 'A3':(10, 7.1), 'B':(12, 7.1), 'C':(14, 7.1)}},
    450: {'t': 80, 'wires': {'A1':(10, 7.1), 'A2':(10, 7.1), 'A3':(12, 7.1), 'B':(14, 7.1), 'C':(16, 7.1)}},
    500: {'t': 90, 'wires': {'A1':(12, 7.1), 'A2':(14, 7.1), 'A3':(16, 7.1), 'B':(12, 9.0), 'C':(16, 9.0)}},
    600: {'t': 100, 'wires': {'A1':(14, 9.0), 'A2':(16, 9.0), 'A3':(20, 9.0), 'B':(20, 10.7), 'C':(24, 10.7)}},
    800: {'t': 120, 'wires': {'A1':(24, 9.0), 'A2':(28, 9.0), 'A3':(32, 9.0), 'B':(24, 10.7), 'C':(30, 10.7)}},
    1000: {'t': 140, 'wires': {'A1':(30, 9.0), 'A2':(34, 9.0), 'A3':(38, 9.0), 'B':(32, 10.7), 'C':(38, 10.7)}},
    1200: {'t': 150, 'wires': {'A1':(36, 9.0), 'A2':(40, 9.0), 'A3':(44, 9.0), 'B':(38, 10.7), 'C':(44, 10.7)}},
}

wf_data_raw = (
    ("WF 100.50.5.7", 100.0, 50.0, 5.0, 7.0, 8.0), ("WF 150.75.5.7", 150.0, 75.0, 5.0, 7.0, 8.0),
    ("WF 200.100.5.5.8", 200.0, 100.0, 5.5, 8.0, 11.0), ("WF 250.125.6.9", 250.0, 125.0, 6.0, 9.0, 12.0),
    ("WF 300.150.6.5.9", 300.0, 150.0, 6.5, 9.0, 13.0), ("WF 350.175.7.11", 350.0, 175.0, 7.0, 11.0, 14.0),
    ("WF 400.200.8.13", 400.0, 200.0, 8.0, 13.0, 16.0), ("WF 450.200.9.14", 450.0, 200.0, 9.0, 14.0, 18.0),
    ("WF 500.200.10.16", 500.0, 200.0, 10.0, 16.0, 20.0), ("WF 600.200.11.17", 600.0, 200.0, 11.0, 17.0, 22.0),
    ("WF 700.300.13.24", 700.0, 300.0, 13.0, 24.0, 28.0), ("WF 800.300.14.26", 800.0, 300.0, 14.0, 26.0, 28.0),
    ("WF 900.300.16.28", 900.0, 300.0, 16.0, 28.0, 28.0)
)
wf_dict = {item[0]: {'H': item[1], 'B': item[2], 'tw': item[3], 'tf': item[4]} for item in wf_data_raw}

FY_WIRE, ES_WIRE, ES_REBAR, EPS_PE, CONC_CU = 1860.0, 200000.0, 200000.0, 0.0055, 0.003

# ==============================================================================
# --- 2. CORE MATH & UNIVERSAL 3D GENERATOR ---
# ==============================================================================
def get_beta1(fc): return 0.85 if fc <= 28 else max(0.65, 0.85 - 0.05 * (fc - 28) / 7)
def get_phi_tied(et): return 0.90 if et >= 0.005 else (0.65 if et <= 0.002 else 0.65 + (et - 0.002) * (0.25 / 0.003))
def get_phi_spiral(et): return 0.90 if et >= 0.005 else (0.75 if et <= 0.002 else 0.75 + (et - 0.002) * (0.15 / 0.003))

def get_rect_rebar_coords(n_col, n_row, B_core, H_core):
    coords = []
    if n_col < 2 or n_row < 2: return coords
    x_vals, y_vals = np.linspace(-B_core/2, B_core/2, n_col), np.linspace(-H_core/2, H_core/2, n_row)
    for x in x_vals: coords.extend([(x, H_core/2), (x, -H_core/2)])
    for y in y_vals[1:-1]: coords.extend([(-B_core/2, y), (B_core/2, y)])
    return coords

def calc_frp_confinement(b, h, rc, fc, Asr, n_ply, tf, Ef, efu, CE, is_circ=False):
    efu_d = CE * efu
    efe = min(0.004, 0.55 * efu_d)
    if is_circ:
        fl = (2 * Ef * n_ply * tf * efe) / b
        ka, kb = 1.0, 1.0
    else:
        fl = (2 * Ef * n_ply * tf * efe) / math.sqrt(b**2 + h**2)
        Ag = b * h
        rho_g = Asr / Ag
        Ae_Ac = max(0.0, min(1.0, (1 - ((b/h)*(h - 2*rc)**2 + (h/b)*(b - 2*rc)**2)/(3*Ag) - rho_g) / (1 - rho_g)))
        ka = Ae_Ac * (min(b, h)/max(b, h))**2
        kb = Ae_Ac * math.sqrt(max(b, h)/min(b, h))
    fcc = fc + 3.3 * ka * fl
    eccu = min(0.01, 0.002 * (1.5 + 12 * kb * (fl / fc) * (efe / 0.002)**0.45))
    return fcc, eccu, fl

def get_fc_frp(strain, fc, fcc, eccu):
    if strain <= 0 or strain > eccu: return 0.0
    Ec = 4700 * math.sqrt(fc)
    E2 = (fcc - fc) / eccu
    if (Ec - E2) <= 0: return fcc
    et_prime = (2 * fc) / (Ec - E2)
    if strain <= et_prime:
        stress = Ec * strain - ((Ec - E2)**2 / (4 * fc)) * (strain**2)
    else:
        stress = fc + E2 * strain
    return max(0.0, min(stress, fcc))

def build_rect_conc_mesh(B, H, grid=22):
    x, y = np.linspace(-B/2, B/2, grid+1), np.linspace(-H/2, H/2, grid+1)
    X, Y = np.meshgrid((x[:-1]+x[1:])/2, (y[:-1]+y[1:])/2)
    return X.ravel(), Y.ravel(), (B/grid)*(H/grid)

def build_circ_conc_mesh(D, grid=22):
    x, y = np.linspace(-D/2, D/2, grid+1), np.linspace(-D/2, D/2, grid+1)
    X, Y = np.meshgrid((x[:-1]+x[1:])/2, (y[:-1]+y[1:])/2)
    X, Y = X.ravel(), Y.ravel()
    mask = X**2 + Y**2 <= (D/2)**2
    return X[mask], Y[mask], (D/grid)**2

def build_wf_mesh(H_wf, B_wf, tw_wf, tf_wf, grid=20):
    x, y = np.linspace(-B_wf/2, B_wf/2, grid+1), np.linspace(-H_wf/2, H_wf/2, grid+1)
    X, Y = np.meshgrid((x[:-1]+x[1:])/2, (y[:-1]+y[1:])/2)
    X, Y = X.ravel(), Y.ravel()
    mask = (np.abs(Y) >= H_wf/2 - tf_wf) | (np.abs(X) <= tw_wf/2)
    return X[mask], Y[mask], (B_wf/grid)*(H_wf/grid)

def revolve_1d_pm_to_3d(Pn, Mn, Pd, Md, n_theta=24):
    thetas = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    P_surf_n, P_surf_d = np.tile(Pn, (n_theta, 1)).T, np.tile(Pd, (n_theta, 1)).T
    Mx_surf_n, My_surf_n = np.outer(Mn, np.cos(thetas)), np.outer(Mn, np.sin(thetas))
    Mx_surf_d, My_surf_d = np.outer(Md, np.cos(thetas)), np.outer(Md, np.sin(thetas))
    P_surf_n, Mx_surf_n, My_surf_n = np.column_stack((P_surf_n, P_surf_n[:, 0])), np.column_stack((Mx_surf_n, Mx_surf_n[:, 0])), np.column_stack((My_surf_n, My_surf_n[:, 0]))
    P_surf_d, Mx_surf_d, My_surf_d = np.column_stack((P_surf_d, P_surf_d[:, 0])), np.column_stack((Mx_surf_d, Mx_surf_d[:, 0])), np.column_stack((My_surf_d, My_surf_d[:, 0]))
    return P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d

def evaluate_3d_mesh(X_conc, Y_conc, A_conc_fiber, X_reb, Y_reb, A_reb_fiber, fy_reb,
                     X_wf, Y_wf, A_wf_fiber, fy_wf, fc, is_frp, fcc, eccu, beta1, cu, Es, phi_func, n_theta=20, n_c=25):
    thetas = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    P_surf_n, Mx_surf_n, My_surf_n = np.zeros((n_c, n_theta)), np.zeros((n_c, n_theta)), np.zeros((n_c, n_theta))
    P_surf_d, Mx_surf_d, My_surf_d = np.zeros((n_c, n_theta)), np.zeros((n_c, n_theta)), np.zeros((n_c, n_theta))

    for j, theta in enumerate(thetas):
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        dist_conc = X_conc * cos_t + Y_conc * sin_t if len(X_conc) > 0 else np.array([])
        dist_reb = X_reb * cos_t + Y_reb * sin_t if len(X_reb) > 0 else np.array([])
        dist_wf = X_wf * cos_t + Y_wf * sin_t if len(X_wf) > 0 else np.array([])

        max_dist = max([np.max(d) for d in [dist_conc, dist_reb, dist_wf] if len(d) > 0] + [-1e9])
        c_vals = np.concatenate([np.linspace(max_dist*10, max_dist, n_c//3), np.linspace(max_dist, 0.001, n_c - n_c//3)]) if max_dist>0 else []

        for i, c in enumerate(c_vals):
            if c <= 0: continue
            a = beta1 * c
            P_tot, Mx_tot, My_tot, min_strain_reb = 0.0, 0.0, 0.0, 0.0

            if len(dist_conc) > 0:
                strain_conc = cu * (dist_conc - (max_dist - c)) / c
                if is_frp:
                    Ec, E2 = 4700 * math.sqrt(fc), ((fcc - fc) / eccu if eccu > 0 else 0)
                    et_prime = (2 * fc) / (Ec - E2) if (Ec - E2) > 0 else 0
                    fc_str = np.zeros_like(strain_conc)
                    c1, c2 = (strain_conc > 0) & (strain_conc <= et_prime), (strain_conc > et_prime) & (strain_conc <= eccu)
                    if (Ec - E2) <= 0: fc_str[(strain_conc > 0) & (strain_conc <= eccu)] = fcc
                    else:
                        fc_str[c1] = Ec * strain_conc[c1] - ((Ec - E2)**2 / (4 * fc)) * (strain_conc[c1]**2)
                        fc_str[c2] = fc + E2 * strain_conc[c2]
                    fc_stress = np.where(max_dist - dist_conc <= a, np.clip(fc_str, 0.0, fcc), 0.0)
                else:
                    fc_stress = np.where((max_dist - dist_conc <= a) & (strain_conc > 0), 0.85 * fc, 0.0)

                F_c = fc_stress * A_conc_fiber
                P_tot += np.sum(F_c); Mx_tot += np.sum(F_c * Y_conc); My_tot += np.sum(F_c * X_conc)

            if len(dist_reb) > 0:
                strain_reb = cu * (dist_reb - (max_dist - c)) / c
                min_strain_reb = np.min(strain_reb)
                fs_reb = np.clip(strain_reb * Es, -fy_reb, fy_reb)
                fs_reb[(max_dist - dist_reb <= a) & (strain_reb > 0)] -= 0.85 * fc
                F_reb = fs_reb * A_reb_fiber
                P_tot += np.sum(F_reb); Mx_tot += np.sum(F_reb * Y_reb); My_tot += np.sum(F_reb * X_reb)

            if len(dist_wf) > 0:
                strain_wf = cu * (dist_wf - (max_dist - c)) / c
                fs_wf = np.clip(strain_wf * Es, -fy_wf, fy_wf)
                fs_wf[(max_dist - dist_wf <= a) & (strain_wf > 0)] -= 0.85 * fc
                F_wf = fs_wf * A_wf_fiber
                P_tot += np.sum(F_wf); Mx_tot += np.sum(F_wf * Y_wf); My_tot += np.sum(F_wf * X_wf)

            phi = phi_func(np.abs(min_strain_reb) if min_strain_reb < 0 else 0)
            P_surf_n[i, j], Mx_surf_n[i, j], My_surf_n[i, j] = P_tot / 1000.0, Mx_tot / 1e6, My_tot / 1e6
            P_surf_d[i, j], Mx_surf_d[i, j], My_surf_d[i, j] = P_tot * phi / 1000.0, Mx_tot * phi / 1e6, My_tot * phi / 1e6

    P_surf_n, Mx_surf_n, My_surf_n = np.column_stack((P_surf_n, P_surf_n[:, 0])), np.column_stack((Mx_surf_n, Mx_surf_n[:, 0])), np.column_stack((My_surf_n, My_surf_n[:, 0]))
    P_surf_d, Mx_surf_d, My_surf_d = np.column_stack((P_surf_d, P_surf_d[:, 0])), np.column_stack((Mx_surf_d, Mx_surf_d[:, 0])), np.column_stack((My_surf_d, My_surf_d[:, 0]))
    return P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d

# --- 1D GENERATORS ---
def generate_cfst_1d(D, t, fc, fy_t, n_bar, d_bar, fy_r, cover, d_tie, Es):
    R, Rc = D / 2.0, D / 2.0 - t
    Rs = R - cover - d_tie - d_bar / 2.0
    A_bar = 0.25 * math.pi * d_bar**2
    y_rebars = [Rs * math.cos(i * 2 * math.pi / n_bar) for i in range(n_bar)] if n_bar > 0 else []
    y_edges = np.linspace(-R, R, 301)
    dy, beta1, cu = 2*R/300, get_beta1(fc), CONC_CU
    Pn, Mn, Pd, Md = [], [], [], []
    for c in np.concatenate([np.linspace(D*5, D, 30), np.linspace(D, 0.1*D, 50), np.linspace(0.1*D, 0.001*D, 30)]):
        a, P_tot, M_tot, net_et = beta1 * c, 0.0, 0.0, 0.0
        for i in range(300):
            y_mid = (y_edges[i] + y_edges[i+1]) / 2.0
            strain = cu * ((y_mid - (R - c)) / c) if c > 0 else 0
            if i == 0: net_et = strain
            A_conc = 2 * np.sqrt(max(0, Rc**2 - y_mid**2)) * dy
            A_steel = 2 * np.sqrt(max(0, R**2 - y_mid**2)) * dy - A_conc
            fs_tube = max(-fy_t, min(fy_t, strain * Es))
            fc_stress = 0.85 * fc if (R - y_mid <= a and strain > 0) else 0.0
            P_tot += fs_tube * A_steel + fc_stress * A_conc
            M_tot += (fs_tube * A_steel + fc_stress * A_conc) * y_mid
        for y_s in y_rebars:
            strain_s = cu * ((y_s - (R - c)) / c) if c > 0 else 0
            fs_rebar = max(-fy_r, min(fy_r, strain_s * Es))
            if R - y_s <= a and strain_s > 0: fs_rebar -= 0.85 * fc
            P_tot += fs_rebar * A_bar; M_tot += fs_rebar * A_bar * y_s
        phi = get_phi_spiral(abs(net_et))
        Pn.append(P_tot/1000.0); Mn.append(abs(M_tot)/1e6)
        Pd.append(P_tot*phi/1000.0); Md.append(abs(M_tot)*phi/1e6)
    Pt = -(math.pi*(R**2 - Rc**2)*fy_t + n_bar*A_bar*fy_r)/1000.0
    return np.append(Pn, Pt), np.append(Mn, 0.0), np.append(Pd, 0.9*Pt), np.append(Md, 0.0)

def generate_spunpile_1d(D, t, fc, n_bar, d_bar, n_is, d_is, fcinf, fyinf):
    R, R_in = D/2.0, D/2.0 - t
    Rs = R - t/2.0
    A_bar = 0.25 * math.pi * d_bar**2
    y_rebars = [Rs * math.cos(i * 2 * math.pi / n_bar) for i in range(n_bar)]
    y_is = [(R_in - 40) * math.cos(i * 2 * math.pi / n_is) for i in range(n_is)] if n_is > 0 else []
    A_is = 0.25 * math.pi * d_is**2 if d_is > 0 else 0
    y_edges = np.linspace(-R, R, 401)
    dy, beta1, cu = 2*R/400, get_beta1(fc), 0.004
    Pn, Mn, Pd, Md = [], [], [], []
    for c in np.concatenate([np.linspace(D*15, D, 50), np.linspace(D, 0.05*D, 80), np.linspace(0.05*D, 0.001*D, 50)]):
        a, P_tot, M_tot, net_et = beta1 * c, 0.0, 0.0, 0.0
        for i in range(400):
            y_mid = (y_edges[i] + y_edges[i+1]) / 2.0
            strain = cu * ((y_mid - (R - c)) / c) if c > 0 else 0
            if i == 0: net_et = strain
            Ac_spun = (2*np.sqrt(max(0, R**2 - y_mid**2)) - 2*np.sqrt(max(0, R_in**2 - y_mid**2))) * dy
            Ac_inf = 2*np.sqrt(max(0, R_in**2 - y_mid**2)) * dy if n_is > 0 else 0
            Fc = (0.85*fc if (R - y_mid <= a and strain > 0) else 0.0)*Ac_spun + (0.85*fcinf if (R - y_mid <= a and strain > 0) else 0.0)*Ac_inf
            P_tot += Fc; M_tot += Fc * y_mid
        for y_s in y_rebars:
            strain_s = (cu * ((y_s - (R - c)) / c) if c > 0 else 0) - EPS_PE
            fs_w = max(-FY_WIRE, min(FY_WIRE, strain_s * ES_WIRE))
            if R - y_s <= a and (strain_s + EPS_PE) > 0: fs_w -= 0.85 * fc
            P_tot += fs_w * A_bar; M_tot += fs_w * A_bar * y_s
        for yi in y_is:
            str_is = cu * ((yi - (R - c)) / c) if c > 0 else 0
            fs_i = max(-fyinf, min(fyinf, str_is * ES_REBAR))
            if R - yi <= a and str_is > 0: fs_i -= 0.85 * fcinf
            P_tot += fs_i * A_is; M_tot += fs_i * A_is * yi
        phi = get_phi_spiral(abs(net_et))
        Pn.append(P_tot/1000.0); Mn.append(abs(M_tot)/1e6)
        Pd.append(P_tot*phi/1000.0); Md.append(abs(M_tot)*phi/1e6)
    Pt = -(n_bar * A_bar * FY_WIRE + n_is * A_is * fyinf)/1000.0
    return np.append(Pn, Pt), np.append(Mn, 0.0), np.append(Pd, 0.9*Pt), np.append(Md, 0.0)

def generate_rc_1d(D, fc, n_bar, d_bar, fy, cov, dtie, is_frp=False, fcc=0, eccu=0):
    R = D/2.0
    Rs = R - cov - dtie - d_bar/2.0
    A_bar = 0.25 * math.pi * d_bar**2
    y_reb = [Rs * math.cos(i * 2 * math.pi / n_bar) for i in range(n_bar)] if n_bar > 0 else []
    y_edges = np.linspace(-R, R, 301)
    dy, beta1, cu = 2*R/300, get_beta1(fc), (eccu if is_frp else CONC_CU)
    Pn, Mn, Pd, Md = [], [], [], []
    for c in np.concatenate([np.linspace(D*5, D, 30), np.linspace(D, 0.1*D, 50), np.linspace(0.1*D, 0.001*D, 30)]):
        a, P_tot, M_tot, net_et = beta1 * c, 0.0, 0.0, 0.0
        for i in range(300):
            y_mid = (y_edges[i] + y_edges[i+1]) / 2.0
            strain = cu * ((y_mid - (R - c)) / c) if c > 0 else 0
            if i == 0: net_et = strain
            Ac = 2 * np.sqrt(max(0, R**2 - y_mid**2)) * dy
            fcs = get_fc_frp(strain, fc, fcc, eccu) if is_frp else (0.85*fc if R-y_mid <= a and strain > 0 else 0)
            P_tot += fcs*Ac; M_tot += fcs*Ac*y_mid
        for y_s in y_reb:
            str_s = cu * ((y_s - (R - c)) / c) if c > 0 else 0
            fs = max(-fy, min(fy, str_s * ES_REBAR))
            if is_frp and str_s > 0: fs -= get_fc_frp(str_s, fc, fcc, eccu)
            elif not is_frp and R-y_s <= a and str_s > 0: fs -= 0.85 * fc
            P_tot += fs*A_bar; M_tot += fs*A_bar*y_s
        phi = get_phi_spiral(abs(net_et))
        Pn.append(P_tot/1000.0); Mn.append(abs(M_tot)/1e6)
        Pd.append(P_tot*phi/1000.0); Md.append(abs(M_tot)*phi/1e6)
    Pt = -(n_bar * A_bar * fy)/1000.0
    return np.append(Pn, Pt), np.append(Mn, 0.0), np.append(Pd, 0.9*Pt), np.append(Md, 0.0)

def generate_rc_rect_1d(B, H, fc, n_col, n_row, d_bar, fy, cov, dtie, is_frp=False, fcc=0, eccu=0):
    Bcore, Hcore = B-2*cov-2*dtie-d_bar, H-2*cov-2*dtie-d_bar
    A_bar = 0.25 * math.pi * d_bar**2
    y_reb = [coord[1] for coord in get_rect_rebar_coords(n_col, n_row, Bcore, Hcore)]
    y_edges = np.linspace(-H/2, H/2, 301)
    dy, beta1, cu = H/300, get_beta1(fc), (eccu if is_frp else CONC_CU)
    Pn, Mn, Pd, Md = [], [], [], []
    for c in np.concatenate([np.linspace(H*5, H, 30), np.linspace(H, 0.1*H, 50), np.linspace(0.1*H, 0.001*H, 30)]):
        a, P_tot, M_tot, net_et = beta1 * c, 0.0, 0.0, 0.0
        for i in range(300):
            y_mid = (y_edges[i] + y_edges[i+1]) / 2.0
            strain = cu * ((y_mid - (H/2 - c)) / c) if c > 0 else 0
            if i == 0: net_et = strain
            fcs = get_fc_frp(strain, fc, fcc, eccu) if is_frp else (0.85*fc if H/2-y_mid <= a and strain > 0 else 0)
            P_tot += fcs*(B*dy); M_tot += fcs*(B*dy)*y_mid
        for y_s in y_reb:
            str_s = cu * ((y_s - (H/2 - c)) / c) if c > 0 else 0
            fs = max(-fy, min(fy, str_s * ES_REBAR))
            if is_frp and str_s > 0: fs -= get_fc_frp(str_s, fc, fcc, eccu)
            elif not is_frp and H/2-y_s <= a and str_s > 0: fs -= 0.85 * fc
            P_tot += fs*A_bar; M_tot += fs*A_bar*y_s
        phi = get_phi_tied(abs(net_et))
        Pn.append(P_tot/1000.0); Mn.append(abs(M_tot)/1e6)
        Pd.append(P_tot*phi/1000.0); Md.append(abs(M_tot)*phi/1e6)
    Pt = -(len(y_reb) * A_bar * fy)/1000.0
    return np.append(Pn, Pt), np.append(Mn, 0.0), np.append(Pd, 0.9*Pt), np.append(Md, 0.0)

# ==============================================================================
# --- 3. DRAWING SECTIONS ---
# ==============================================================================

def draw_rc_rect_section(ax, B, H, n_col, n_row, d_bar, cov, d_tie, is_frp=False, n_ply=0, t_ply=0, rc=0):
    ax.set_aspect('equal'); ax.axis('off')
    # 1. Visualisasi FRP Wrap (jika ada)
    if is_frp:
        tw = max(5, n_ply * t_ply * 3)
        ax.add_patch(patches.FancyBboxPatch((-B/2-tw, -H/2-tw), B+2*tw, H+2*tw,
            boxstyle=f"round,pad=0,rounding_size={rc+tw}", fc='#FFCA28', ec='#FF8F00', lw=2.0))

    # 2. Visualisasi Kolom (dengan rounding jika FRP aktif)
    ax.add_patch(patches.FancyBboxPatch((-B/2, -H/2), B, H,
        boxstyle=f"round,pad=0,rounding_size={rc if is_frp else 0}", fc='#E3F2FD', ec='#1565C0', lw=1.5))

    # 3. Visualisasi Sengkang (Tie)
    ax.add_patch(patches.Rectangle((-B/2 + cov, -H/2 + cov), B - 2*cov, H - 2*cov,
        fill=False, ec='#F57C00', ls='--', lw=1.8))

    # 4. Visualisasi Tulangan Utama
    B_core, H_core = B - 2*cov - 2*d_tie - d_bar, H - 2*cov - 2*d_tie - d_bar
    for (x, y) in get_rect_rebar_coords(n_col, n_row, B_core, H_core):
        ax.add_patch(patches.Circle((x, y), d_bar/2, fc='#C62828', ec='#B71C1C'))

    # 5. Axis Styling
    max_dim = max(B, H)
    ax.plot([-max_dim*0.6, max_dim*0.6], [0, 0], c='#9E9E9E', ls='-.', lw=1.0)
    ax.plot([0, 0], [-max_dim*0.6, max_dim*0.6], c='#9E9E9E', ls='-.', lw=1.0)
    ax.set_xlim(-max_dim*0.6, max_dim*0.6); ax.set_ylim(-max_dim*0.6, max_dim*0.6)
def draw_rect_section(ax, B, H, n_col, n_row, d_bar, cov, d_tie, is_frp=False, n_ply=0, t_ply=0, rc=0, is_src=False, Hw=0, Bw=0, tww=0, tfw=0):
    ax.set_aspect('equal'); ax.axis('off')
    if is_frp:
        tw = max(5, n_ply*t_ply*3)
        ax.add_patch(patches.FancyBboxPatch((-B/2-tw, -H/2-tw), B+2*tw, H+2*tw, boxstyle=f"round,pad=0,rounding_size={rc+tw}", fc='#FFCA28', ec='#FF8F00', lw=2))
    ax.add_patch(patches.FancyBboxPatch((-B/2, -H/2), B, H, boxstyle=f"round,pad=0,rounding_size={rc if is_frp else 0}", fc='#E3F2FD', ec='#1565C0', lw=1.5))
    ax.add_patch(patches.Rectangle((-B/2+cov, -H/2+cov), B-2*cov, H-2*cov, fill=False, ec='#F57C00', ls='--', lw=1.8))
    for (x, y) in get_rect_rebar_coords(n_col, n_row, B-2*cov-2*d_tie-d_bar, H-2*cov-2*d_tie-d_bar):
        ax.add_patch(patches.Circle((x, y), d_bar/2, fc='#C62828', ec='#B71C1C'))
    if is_src:
        pts = [(-Bw/2, Hw/2), (Bw/2, Hw/2), (Bw/2, Hw/2-tfw), (tww/2, Hw/2-tfw), (tww/2, -Hw/2+tfw), (Bw/2, -Hw/2+tfw), (Bw/2, -Hw/2), (-Bw/2, -Hw/2), (-Bw/2, -Hw/2+tfw), (-tww/2, -Hw/2+tfw), (-tww/2, Hw/2-tfw), (-Bw/2, Hw/2-tfw)]
        ax.add_patch(patches.Polygon(pts, fc='#5C6BC0', ec='#283593', lw=1.5))
    md = max(B, H)
    ax.plot([-md*0.6, md*0.6], [0, 0], c='#9E9E9E', ls='-.', lw=1); ax.plot([0, 0], [-md*0.6, md*0.6], c='#9E9E9E', ls='-.', lw=1)
    ax.set_xlim(-md*0.6, md*0.6); ax.set_ylim(-md*0.6, md*0.6)

def draw_circ_section(ax, D, n_bar, d_bar, cov, d_tie, is_frp=False, n_ply=0, t_ply=0, is_cfst=False, t_tube=0, is_spun=False, nis=0, dis=0, is_src=False, Hw=0, Bw=0, tww=0, tfw=0):
    ax.set_aspect('equal'); ax.axis('off')
    R = D/2
    if is_frp:
        ax.add_patch(patches.Circle((0, 0), R + max(5, n_ply*t_ply*3), fc='#FFCA28', ec='#FF8F00', lw=2))
    ax.add_patch(patches.Circle((0, 0), R, fc='#E3F2FD', ec='#1565C0', lw=1.5))

    if is_cfst:
        ax.add_patch(patches.Circle((0, 0), R-t_tube, fc='#F8F9FA', ec='#1565C0', lw=1.5))
        ax.add_patch(patches.Circle((0, 0), R-cov-d_tie/2, fill=False, ec='#F57C00', ls='--', lw=1.8))
        for i in range(n_bar): ax.add_patch(patches.Circle(((R-cov-d_tie-d_bar/2)*math.cos(i*2*np.pi/n_bar), (R-cov-d_tie-d_bar/2)*math.sin(i*2*np.pi/n_bar)), d_bar/2, fc='#C62828', ec='#B71C1C'))
    elif is_spun:
        ax.add_patch(patches.Circle((0, 0), R-t_tube, fc=('#BBDEFB' if nis>0 else 'white'), ec='#1976D2', lw=1.5))
        for i in range(n_bar): ax.add_patch(patches.Circle(((R-t_tube/2)*math.cos(i*2*np.pi/n_bar), (R-t_tube/2)*math.sin(i*2*np.pi/n_bar)), d_bar/2, fc='#283593', ec='#1A237E'))
        if nis > 0:
            for i in range(nis): ax.add_patch(patches.Circle((max(0, R-t_tube-40)*math.cos(i*2*np.pi/nis), max(0, R-t_tube-40)*math.sin(i*2*np.pi/nis)), dis/2, fc='#C62828', ec='#B71C1C', lw=0.5))
    else:
        ax.add_patch(patches.Circle((0, 0), R-cov-d_tie/2, fill=False, ec='#F57C00', ls='--', lw=1.8))
        for i in range(n_bar): ax.add_patch(patches.Circle(((R-cov-d_tie-d_bar/2)*math.cos(i*2*np.pi/n_bar), (R-cov-d_tie-d_bar/2)*math.sin(i*2*np.pi/n_bar)), d_bar/2, fc='#C62828', ec='#B71C1C'))
        if is_src:
            pts = [(-Bw/2, Hw/2), (Bw/2, Hw/2), (Bw/2, Hw/2-tfw), (tww/2, Hw/2-tfw), (tww/2, -Hw/2+tfw), (Bw/2, -Hw/2+tfw), (Bw/2, -Hw/2), (-Bw/2, -Hw/2), (-Bw/2, -Hw/2+tfw), (-tww/2, -Hw/2+tfw), (-tww/2, Hw/2-tfw), (-Bw/2, Hw/2-tfw)]
            ax.add_patch(patches.Polygon(pts, fc='#5C6BC0', ec='#283593', lw=1.5))

    ax.plot([-D*0.6, D*0.6], [0, 0], c='#9E9E9E', ls='-.', lw=1); ax.plot([0, 0], [-D*0.6, D*0.6], c='#9E9E9E', ls='-.', lw=1)
    ax.set_xlim(-D*0.7, D*0.7); ax.set_ylim(-D*0.7, D*0.7)


# ==============================================================================
# --- 4. UI WIDGETS ---
# ==============================================================================
analisa_dropdown = widgets.Dropdown(
    options=[
        '1. Baja Terisi Beton (CFST)', '2. Spun Pile (Standar)', '3. Bore Pile (RC Bulat)',
        '4. Bore Pile Terisi WF (SRC Bulat)', '5. Kolom Persegi Terisi WF (SRC)',
        '6. Spun Pile + Infill Beton', '7. Kolom Persegi (RC Rectangular)',
        '8. Kolom Bulat (RC) + FRP Wrap', '9. Kolom Persegi (RC) + FRP Wrap'
    ], value='1. Baja Terisi Beton (CFST)', description='Mode Analisis:', layout=widgets.Layout(width='320px', margin='0px 0px 10px 0px')
)

l = widgets.Layout(width='170px')
w_D, w_B, w_H, w_fc = widgets.FloatText(value=600, step=50, description='Diameter:', layout=l), widgets.FloatText(value=400, step=50, description='Lebar B:', layout=l), widgets.FloatText(value=600, step=50, description='Panjang H:', layout=l), widgets.FloatText(value=35, step=5, description="fc' (MPa):", layout=l)
w_n_col, w_n_row, w_n_rebar, w_d_rebar = widgets.IntText(value=4, description='n-Sumbu X:', layout=l), widgets.IntText(value=5, description='n-Sumbu Y:', layout=l), widgets.IntText(value=12, description='Jml Tul:', layout=l), widgets.Dropdown(options=[10, 13, 16, 19, 22, 25, 29, 32], value=22, description='D. Rebar:', layout=l)
w_fy_rebar, w_cover, w_s_tie, w_fy_tie = widgets.FloatText(value=420, step=10, description='fy Rebar:', layout=l), widgets.FloatText(value=40, step=5, description='Cover (mm):', layout=l), widgets.FloatText(value=100, step=10, description='Spasi Tie:', layout=l), widgets.FloatText(value=420, step=10, description='fyt Tie:', layout=l)
w_frp_n, w_frp_tf, w_frp_Ef, w_frp_efu, w_frp_ce, w_frp_rc = widgets.IntText(value=6, description='FRP Ply (n):', layout=l), widgets.FloatText(value=0.33, step=0.01, description='FRP t/ply:', layout=l), widgets.FloatText(value=227527, step=1000, description='FRP E (MPa):', layout=l), widgets.FloatText(value=0.0167, step=0.001, description='FRP \u03B5_fu:', layout=l), widgets.FloatText(value=0.95, step=0.05, description='FRP C_E:', layout=l), widgets.FloatText(value=25, step=5, description='Corner rc:', layout=l)
w_t_tube, w_fy_tube = widgets.FloatText(value=12, step=1, description='t Baja (mm):', layout=l), widgets.FloatText(value=345, step=10, description='fy Baja:', layout=l)
w_spun_D, w_spun_cls, w_fc_spun = widgets.Dropdown(options=list(SPUN_PILE_DATA.keys()), value=600, description='D. SpunPile:', layout=l), widgets.Dropdown(options=['A1', 'A2', 'A3', 'B', 'C'], value='C', description='Kls SpunPile:', layout=l), widgets.FloatText(value=52, step=5, description="Spun fc':", layout=l)
w_n_isian, w_d_isian, w_fc_infill, w_fy_infill = widgets.IntText(value=8, description='n Isian:', layout=l), widgets.Dropdown(options=[0, 10, 13, 16, 19, 22, 25, 29, 32], value=16, description='D Isian:', layout=l), widgets.FloatText(value=30, step=5, description="Infill fc':", layout=l), widgets.FloatText(value=420, step=10, description='fy Infill:', layout=l)
w_wf_drop, w_fy_wf = widgets.Dropdown(options=list(wf_dict.keys()), value='WF 300.150.6.5.9', description='Pilih WF:', layout=widgets.Layout(width='200px')), widgets.FloatText(value=250, step=10, description='fy WF:', layout=l)

global_upload_btn = widgets.FileUpload(accept='.csv', multiple=False, description='Upload Loads (CSV)', button_style='info')

ui_panel = widgets.VBox(layout=widgets.Layout(overflow='hidden'))
output_area = widgets.Output(layout=widgets.Layout(overflow='hidden'))

def update_ui_visibility(*args):
    mode = analisa_dropdown.value
    rows = []
    if mode == '1. Baja Terisi Beton (CFST)':
        rows = [widgets.HBox([w_D, w_t_tube, w_fc, w_fy_tube]), widgets.HBox([w_n_rebar, w_d_rebar, w_fy_rebar, w_cover]), widgets.HBox([w_s_tie, w_fy_tie, global_upload_btn])]
    elif mode == '2. Spun Pile (Standar)':
        rows = [widgets.HBox([w_spun_D, w_spun_cls, w_fc_spun]), widgets.HBox([global_upload_btn])]
    elif mode == '3. Bore Pile (RC Bulat)':
        rows = [widgets.HBox([w_D, w_fc, w_cover]), widgets.HBox([w_n_rebar, w_d_rebar, w_fy_rebar]), widgets.HBox([w_s_tie, w_fy_tie, global_upload_btn])]
    elif mode == '4. Bore Pile Terisi WF (SRC Bulat)':
        rows = [widgets.HBox([w_D, w_fc, w_cover]), widgets.HBox([w_n_rebar, w_d_rebar, w_fy_rebar]), widgets.HBox([w_wf_drop, w_fy_wf, w_s_tie, w_fy_tie]), widgets.HBox([global_upload_btn])]
    elif mode == '5. Kolom Persegi Terisi WF (SRC)':
        rows = [widgets.HBox([w_B, w_H, w_fc, w_cover]), widgets.HBox([w_n_col, w_n_row, w_d_rebar, w_fy_rebar]), widgets.HBox([w_wf_drop, w_fy_wf, w_s_tie, w_fy_tie]), widgets.HBox([global_upload_btn])]
    elif mode == '6. Spun Pile + Infill Beton':
        rows = [widgets.HBox([w_spun_D, w_spun_cls, w_fc_spun]), widgets.HBox([w_n_isian, w_d_isian, w_fc_infill, w_fy_infill]), widgets.HBox([global_upload_btn])]
    elif mode == '7. Kolom Persegi (RC Rectangular)':
        rows = [widgets.HBox([w_B, w_H, w_fc, w_cover]), widgets.HBox([w_n_col, w_n_row, w_d_rebar, w_fy_rebar]), widgets.HBox([w_s_tie, w_fy_tie, global_upload_btn])]
    elif mode == '8. Kolom Bulat (RC) + FRP Wrap':
        rows = [widgets.HBox([w_D, w_fc, w_cover]), widgets.HBox([w_n_rebar, w_d_rebar, w_fy_rebar]), widgets.HBox([w_s_tie, w_fy_tie]), widgets.HBox([w_frp_n, w_frp_tf, w_frp_Ef, w_frp_efu, w_frp_ce]), widgets.HBox([global_upload_btn])]
    elif mode == '9. Kolom Persegi (RC) + FRP Wrap':
        rows = [widgets.HBox([w_B, w_H, w_fc, w_cover]), widgets.HBox([w_n_col, w_n_row, w_d_rebar, w_fy_rebar]), widgets.HBox([w_s_tie, w_fy_tie]), widgets.HBox([w_frp_n, w_frp_tf, w_frp_Ef, w_frp_efu, w_frp_ce, w_frp_rc]), widgets.HBox([global_upload_btn])]
    ui_panel.children = rows
    render_dashboard()

analisa_dropdown.observe(update_ui_visibility, names='value')
for w in [w_D, w_B, w_H, w_fc, w_n_col, w_n_row, w_n_rebar, w_d_rebar, w_fy_rebar, w_cover, w_s_tie, w_fy_tie, w_t_tube, w_fy_tube, w_spun_D, w_spun_cls, w_fc_spun, w_n_isian, w_d_isian, w_fc_infill, w_fy_infill, w_wf_drop, w_fy_wf, w_frp_n, w_frp_tf, w_frp_Ef, w_frp_efu, w_frp_ce, w_frp_rc, global_upload_btn]:
    w.observe(lambda x: render_dashboard(), names='value')

# ==============================================================================
# --- 5. DASHBOARD RENDERER ---
# ==============================================================================
def render_dashboard(*args):
    with output_area:
        clear_output(wait=True)
        mode = analisa_dropdown.value

        # Load CSV
        mx_vals, my_vals, pu_vals = [], [], []
        if global_upload_btn.value:
            try:
                content = list(global_upload_btn.value.values())[0]['content'] if isinstance(global_upload_btn.value, dict) else global_upload_btn.value[0]['content']
                df = pd.read_csv(io.BytesIO(content), header=None, sep=r'[;,]', engine='python')
                if isinstance(df.iloc[0, 0], str) and not df.iloc[0, 0].replace('.', '', 1).replace('-', '', 1).isdigit(): df = df.iloc[1:]
                data = df.astype(float)
                pu_vals, mx_vals, my_vals = data.iloc[:, 0].values, data.iloc[:, 1].values, data.iloc[:, 2].values
            except Exception as e: print(f"Error membaca CSV: {e}")

        # SETUP CANVAS
        fig = plt.figure(figsize=(18, 10.5), dpi=100)
        fig.patch.set_facecolor('white')
        gs = fig.add_gridspec(2, 3, height_ratios=[0.3, 1], width_ratios=[1, 1.2, 1.4])

        ax_info = fig.add_subplot(gs[0, :]); ax_info.axis('off')
        ax_info.add_patch(patches.Rectangle((0.02, 0.05), 0.96, 0.90, lw=1.2, ec='#e0e0e0', fc='#F8F9FA'))
        ax_geom = fig.add_subplot(gs[1, 0])
        ax_geom.set_title("CROSS SECTION", fontsize=12, fontweight='bold', pad=10, color='#1565C0')
        ax_pm = fig.add_subplot(gs[1, 1])
        ax_3d = fig.add_subplot(gs[1, 2], projection='3d'); ax_3d.view_init(elev=20, azim=-45)

        def get_dtie(D_core, Ag, Ach, fc, fyt, s_tie):
            rho_s_req = max(0.45 * ((Ag / Ach) - 1) * (fc / fyt), 0.12 * (fc / fyt), 0.001)
            return next((d for d in [10, 13, 16, 19, 22, 25] if d >= math.sqrt((rho_s_req * D_core * s_tie) / math.pi)), 25)

        tit_t, dim_t, fc_t, po_t, reb_t, stl_t, mcr_t, r1_t, r2_t, r3_t, ei_u_t, ei_c_t, dc_t = [""]*13
        M_n_full, P_n_full, M_d_full, P_d_full = [], [], [], []

        # ROUTING & CALCULATIONS
        if mode == '1. Baja Terisi Beton (CFST)':
            D, t, fc, fytb, n_b, d_b, fyr, cov, stie, fytie = w_D.value, w_t_tube.value, w_fc.value, w_fy_tube.value, w_n_rebar.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value
            dtu = get_dtie(D - 2*cov, 0.25*math.pi*D**2, 0.25*math.pi*(D-2*cov)**2, fc, fytie, stie)
            Pn, Mn, Pd, Md = generate_cfst_1d(D, t, fc, fytb, n_b, d_b, fyr, cov, dtu, ES_REBAR)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = revolve_1d_pm_to_3d(Pn, Mn, Pd, Md)
            draw_circ_section(ax_geom, D, n_b, d_b, cov, dtu, is_cfst=True, t_tube=t)
            Ec, fr, Ag, Ac, Asr = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), 0.25*math.pi*D**2, 0.25*math.pi*(D-2*t)**2, n_b*0.25*math.pi*d_b**2
            Atube = Ag - Ac
            EI_uncracked = (Ec * ((math.pi*(D-2*t)**4)/64) + ES_REBAR * ((math.pi*D**4)/64 - (math.pi*(D-2*t)**4)/64) + ES_REBAR * (Asr*(D/2-cov-dtu-d_b/2)**2/2)) / 1e9
            EI_cracked = (0.35 * Ec * ((math.pi*(D-2*t)**4)/64) + ES_REBAR * ((math.pi*D**4)/64 - (math.pi*(D-2*t)**4)/64) + ES_REBAR * (Asr*(D/2-cov-dtu-d_b/2)**2/2)) / 1e9
            Po = (0.85*fc*Ac + fytb*Atube + fyr*Asr) / 1000.0
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "KOMPOSIT CFST + TULANGAN", f"D={D} mm, t={t} mm", f"f'c: {fc} MPa", f"Po: {Po:,.1f} kN", f"Rebar: {n_b}-D{d_b} (fy {fyr})", f"Tube: fy {fytb}"
            mcr_t, ei_u_t, ei_c_t, r1_t, r2_t, r3_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%", f"\u03C1 tube: {(Atube/Ag)*100:.2f}%", f"\u03C1 tot: {((Asr+Atube)/Ag)*100:.2f}%"

        elif mode == '2. Spun Pile (Standar)':
            D, cls, fc = w_spun_D.value, w_spun_cls.value, w_fc_spun.value
            t, n_w, d_w = SPUN_PILE_DATA[D]['t'], SPUN_PILE_DATA[D]['wires'][cls][0], SPUN_PILE_DATA[D]['wires'][cls][1]
            Pn, Mn, Pd, Md = generate_spunpile_1d(D, t, fc, n_w, d_w, 0, 0, 0, 0)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = revolve_1d_pm_to_3d(Pn, Mn, Pd, Md)
            draw_circ_section(ax_geom, D, n_w, d_w, 0, 0, is_spun=True, t_tube=t)
            Ec, fr, Ac, Awire = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), 0.25*math.pi*(D**2 - (D-2*t)**2), n_w*0.25*math.pi*d_w**2
            EI_uncracked = (Ec * (math.pi*(D**4 - (D-2*t)**4)/64) + ES_WIRE * (Awire*(D/2-t/2)**2/2)) / 1e9
            EI_cracked = (0.35 * Ec * (math.pi*(D**4 - (D-2*t)**4)/64) + ES_WIRE * (Awire*(D/2-t/2)**2/2)) / 1e9
            Po = (0.85*fc*Ac + FY_WIRE*Awire) / 1000.0
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "TIANG PANCANG SPUN PILE", f"D={D} mm, t={t} mm", f"f'c: {fc} MPa", f"Po: {Po:,.1f} kN", f"Wires: {n_w}-D{d_w} (fy {FY_WIRE})", f"Kelas: {cls}"
            mcr_t, ei_u_t, ei_c_t, r1_t = f"Mcr: {(fr + 1100*Awire/Ac)*(EI_uncracked*1e9/Ec)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 wires: {(Awire/Ac)*100:.2f}% (thd selimut)"

        elif mode == '3. Bore Pile (RC Bulat)':
            D, fc, n_b, d_b, fyr, cov, stie, fytie = w_D.value, w_fc.value, w_n_rebar.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value
            dtu = get_dtie(D - 2*cov, 0.25*math.pi*D**2, 0.25*math.pi*(D-2*cov)**2, fc, fytie, stie)
            Pn, Mn, Pd, Md = generate_rc_1d(D, fc, n_b, d_b, fyr, cov, dtu)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = revolve_1d_pm_to_3d(Pn, Mn, Pd, Md)
            draw_circ_section(ax_geom, D, n_b, d_b, cov, dtu)
            Ec, fr, Ag, Asr = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), 0.25*math.pi*D**2, n_b*0.25*math.pi*d_b**2
            EI_uncracked = (Ec * (math.pi*D**4/64) + ES_REBAR * (Asr*(D/2-cov-dtu-d_b/2)**2/2)) / 1e9
            EI_cracked = (0.35 * Ec * (math.pi*D**4/64) + ES_REBAR * (Asr*(D/2-cov-dtu-d_b/2)**2/2)) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "KOLOM BETON BERTULANG (SPIRAL)", f"D={D} mm", f"f'c: {fc} MPa", f"Po: {(0.85*fc*(Ag-Asr)+fyr*Asr)/1000.0:,.1f} kN", f"Rebar: {n_b}-D{d_b} (fy {fyr})", f"Ties: D{dtu}-{stie}"
            mcr_t, ei_u_t, ei_c_t, r1_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1: {(Asr/Ag)*100:.2f}%"

        elif mode == '4. Bore Pile Terisi WF (SRC Bulat)':
            D, fc, wf, fywf, n_b, d_b, fyr, cov, stie, fytie = w_D.value, w_fc.value, w_wf_drop.value, w_fy_wf.value, w_n_rebar.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value
            dtu, Hwf, Bwf, twwf, tfwf = get_dtie(D - 2*cov, 0.25*math.pi*D**2, 0.25*math.pi*(D-2*cov)**2, fc, fytie, stie), wf_dict[wf]['H'], wf_dict[wf]['B'], wf_dict[wf]['tw'], wf_dict[wf]['tf']
            X_c, Y_c, A_c = build_circ_conc_mesh(D)
            ang = np.linspace(0, 2*np.pi, n_b, endpoint=False)
            X_r, Y_r, A_r = (D/2-cov-dtu-d_b/2)*np.cos(ang), (D/2-cov-dtu-d_b/2)*np.sin(ang), 0.25*math.pi*d_b**2
            X_w, Y_w, A_w = build_wf_mesh(Hwf, Bwf, twwf, tfwf)
            Pn, Mn, Pd, Md = generate_rc_1d(D, fc, n_b, d_b, fyr, cov, dtu) # Fallback 1D needed for 2D curve
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = evaluate_3d_mesh(X_c, Y_c, A_c, X_r, Y_r, A_r, fyr, X_w, Y_w, A_w, fywf, fc, False, 0, 0, get_beta1(fc), CONC_CU, ES_REBAR, get_phi_spiral)
            draw_circ_section(ax_geom, D, n_b, d_b, cov, dtu, is_src=True, Hw=Hwf, Bw=Bwf, tww=twwf, tfw=tfwf)
            Ag, Asr, Awf = 0.25*math.pi*D**2, len(X_r)*A_r, len(X_w)*A_w
            Ec, fr = 4700*math.sqrt(fc), 0.62*math.sqrt(fc)
            Ig, Isr, Iwf = (math.pi*D**4)/64, sum([A_r * (D/2-cov-dtu-d_b/2)**2/2 for _ in range(n_b)]), (Bwf*Hwf**3)/12 - ((Bwf-twwf)*(Hwf-2*tfwf)**3)/12
            EI_uncracked = (Ec * Ig + ES_REBAR * Isr + ES_REBAR * Iwf) / 1e9
            EI_cracked = (0.35 * Ec * Ig + ES_REBAR * Isr + ES_REBAR * Iwf) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "KOLOM KOMPOSIT (SRC BUNDAR + WF)", f"D={D} mm", f"f'c: {fc} MPa", f"Po: {(0.85*fc*(Ag-Asr-Awf)+fyr*Asr+fywf*Awf)/1000.0:,.1f} kN", f"Rebar: {n_b}-D{d_b} (fy {fyr})", f"WF: {wf} (fy {fywf})"
            mcr_t, ei_u_t, ei_c_t, r1_t, r2_t, r3_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%", f"\u03C1 WF: {(Awf/Ag)*100:.2f}%", f"\u03C1 tot: {((Asr+Awf)/Ag)*100:.2f}%"

        elif mode == '5. Kolom Persegi Terisi WF (SRC)':
            B, H, fc, wf, fywf, nc, nr, d_b, fyr, cov, stie, fytie = w_B.value, w_H.value, w_fc.value, w_wf_drop.value, w_fy_wf.value, w_n_col.value, w_n_row.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value
            dtu, Hwf, Bwf, twwf, tfwf = get_dtie(min(B-2*cov, H-2*cov), B*H, (B-2*cov)*(H-2*cov), fc, fytie, stie), wf_dict[wf]['H'], wf_dict[wf]['B'], wf_dict[wf]['tw'], wf_dict[wf]['tf']
            X_c, Y_c, A_c = build_rect_conc_mesh(B, H)
            r_coords = get_rect_rebar_coords(nc, nr, B-2*cov-2*dtu-d_b, H-2*cov-2*dtu-d_b)
            X_r, Y_r, A_r = np.array(r_coords)[:,0], np.array(r_coords)[:,1], 0.25*math.pi*d_b**2
            X_w, Y_w, A_w = build_wf_mesh(Hwf, Bwf, twwf, tfwf)
            Pn, Mn, Pd, Md = generate_rc_rect_1d(B, H, fc, nc, nr, d_b, fyr, cov, dtu)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = evaluate_3d_mesh(X_c, Y_c, A_c, X_r, Y_r, A_r, fyr, X_w, Y_w, A_w, fywf, fc, False, 0, 0, get_beta1(fc), CONC_CU, ES_REBAR, get_phi_tied)
            draw_rect_section(ax_geom, B, H, nc, nr, d_b, cov, dtu, is_src=True, Hw=Hwf, Bw=Bwf, tww=twwf, tfw=tfwf)
            Ag, Asr, Awf = B*H, len(X_r)*A_r, len(X_w)*A_w
            Ec, fr = 4700*math.sqrt(fc), 0.62*math.sqrt(fc)
            Ig, Isr, Iwf = (B*H**3)/12, sum([A_r * y**2 for (_, y) in r_coords]), (Bwf*Hwf**3)/12 - ((Bwf-twwf)*(Hwf-2*tfwf)**3)/12
            EI_uncracked = (Ec * Ig + ES_REBAR * Isr + ES_REBAR * Iwf) / 1e9
            EI_cracked = (0.35 * Ec * Ig + ES_REBAR * Isr + ES_REBAR * Iwf) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "KOLOM KOMPOSIT (SRC PERSEGI + WF)", f"B={B}, H={H} mm", f"f'c: {fc} MPa", f"Po: {(0.85*fc*(Ag-Asr-Awf)+fyr*Asr+fywf*Awf)/1000.0:,.1f} kN", f"Rebar: {len(X_r)}-D{d_b} (fy {fyr})", f"WF: {wf} (fy {fywf})"
            mcr_t, ei_u_t, ei_c_t, r1_t, r2_t, r3_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(H/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%", f"\u03C1 WF: {(Awf/Ag)*100:.2f}%", f"\u03C1 tot: {((Asr+Awf)/Ag)*100:.2f}%"

        elif mode == '6. Spun Pile + Infill Beton':
            D, cls, fc, nis, dis, fcinf, fyinf = w_spun_D.value, w_spun_cls.value, w_fc_spun.value, w_n_isian.value, w_d_isian.value, w_fc_infill.value, w_fy_infill.value
            t, n_w, d_w = SPUN_PILE_DATA[D]['t'], SPUN_PILE_DATA[D]['wires'][cls][0], SPUN_PILE_DATA[D]['wires'][cls][1]
            Pn, Mn, Pd, Md = generate_spunpile_1d(D, t, fc, n_w, d_w, nis, dis, fcinf, fyinf)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = revolve_1d_pm_to_3d(Pn, Mn, Pd, Md)
            _, _, Pdb, Mdb = generate_spunpile_1d(D, t, fc, n_w, d_w, 0, 0, 0, 0)
            M_db_full, P_db_full = np.concatenate([-Mdb[::-1], Mdb]), np.concatenate([Pdb[::-1], Pdb])
            _, _, _, P_surf_db, Mx_surf_db, My_surf_db = revolve_1d_pm_to_3d(Pn, Mn, Pdb, Mdb)
            draw_circ_section(ax_geom, D, n_w, d_w, 0, 0, is_spun=True, t_tube=t, nis=nis, dis=dis)
            Ac, Ain, Aw, Ai = 0.25*math.pi*(D**2-(D-2*t)**2), 0.25*math.pi*(D-2*t)**2, n_w*0.25*math.pi*d_w**2, nis*0.25*math.pi*dis**2
            Ec_spun, Ec_inf, fr = 4700*math.sqrt(fc), 4700*math.sqrt(fcinf), 0.62*math.sqrt(fc)
            Ig, Iinner = (math.pi * D**4)/64, (math.pi * (D - 2*t)**4)/64
            Ip = sum([0.25*math.pi*d_w**2 * (D/2 - t/2 * math.cos(i * 2 * math.pi / n_w))**2 for i in range(n_w)])
            Rs_inf = max(0, D/2 - t - 40)
            I_is = sum([0.25*math.pi*dis**2 * (Rs_inf * math.cos(i * 2 * math.pi / nis))**2 for i in range(nis)]) if nis > 0 else 0
            EI_uncracked = (Ec_spun * (Ig - Iinner) + Ec_inf * Iinner + ES_WIRE * Ip + ES_REBAR * I_is) / 1e9
            EI_cracked = (0.35 * Ec_spun * (Ig - Iinner) + 0.35 * Ec_inf * Iinner + ES_WIRE * Ip + ES_REBAR * I_is) / 1e9
            fpe = (1100 * Aw) / Ac
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "SPUN PILE + INFILL BETON", f"D={D}, t={t} mm", f"fc_S: {fc} | fc_I: {fcinf}", f"Po: {(0.85*fc*Ac+0.85*fcinf*Ain+FY_WIRE*Aw+fyinf*Ai)/1000.0:,.1f} kN", f"Wires: {n_w}-D{d_w}", f"Infill: {nis}-D{dis}"
            mcr_t, ei_u_t, ei_c_t, r1_t, r2_t = f"Mcr: {(fr + fpe)*(EI_uncracked*1e9/Ec_spun)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 wires: {(Aw/Ac)*100:.2f}%", f"\u03C1 infill: {(Ai/Ain)*100:.2f}%" if Ain>0 else ""

        elif mode == '7. Kolom Persegi (RC Rectangular)':
            B, H, fc, nc, nr, d_b, fyr, cov, stie, fytie = w_B.value, w_H.value, w_fc.value, w_n_col.value, w_n_row.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value
            dtu = get_dtie(min(B-2*cov, H-2*cov), B*H, (B-2*cov)*(H-2*cov), fc, fytie, stie)
            X_c, Y_c, A_c = build_rect_conc_mesh(B, H)
            r_coords = get_rect_rebar_coords(nc, nr, B-2*cov-2*dtu-d_b, H-2*cov-2*dtu-d_b)
            X_r, Y_r, A_r = np.array(r_coords)[:,0], np.array(r_coords)[:,1], 0.25*math.pi*d_b**2
            Pn, Mn, Pd, Md = generate_rc_rect_1d(B, H, fc, nc, nr, d_b, fyr, cov, dtu)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = evaluate_3d_mesh(X_c, Y_c, A_c, X_r, Y_r, A_r, fyr, [], [], 0, 0, fc, False, 0, 0, get_beta1(fc), CONC_CU, ES_REBAR, get_phi_tied)
            draw_rc_rect_section(ax_geom, B, H, nc, nr, d_b, cov, dtu)
            Ec, fr, Ag, Asr = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), B*H, len(X_r)*A_r
            Isr = sum([A_r * y**2 for (_, y) in r_coords])
            EI_uncracked = (Ec * (B*H**3/12) + ES_REBAR * Isr) / 1e9
            EI_cracked = (0.35 * Ec * (B*H**3/12) + ES_REBAR * Isr) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, r1_t = "KOLOM BETON BERTULANG (PERSEGI)", f"B={B}, H={H} mm", f"f'c: {fc} MPa", f"Po: {(0.85*fc*(Ag-Asr)+fyr*Asr)/1000.0:,.1f} kN", f"Rebar: {len(X_r)}-D{d_b}", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%"
            mcr_t, ei_u_t, ei_c_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(H/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2"

        elif mode == '8. Kolom Bulat (RC) + FRP Wrap':
            D, fc, n_b, d_b, fyr, cov, stie, fytie, n_ply, tf, Ef, efu, CE = w_D.value, w_fc.value, w_n_rebar.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value, w_frp_n.value, w_frp_tf.value, w_frp_Ef.value, w_frp_efu.value, w_frp_ce.value
            dtu = get_dtie(D - 2*cov, 0.25*math.pi*D**2, 0.25*math.pi*(D-2*cov)**2, fc, fytie, stie)
            Asr = n_b * 0.25*math.pi*d_b**2
            fcc, eccu, fl = calc_frp_confinement(D, 0, 0, fc, Asr, n_ply, tf, Ef, efu, CE, is_circ=True)
            Pn, Mn, Pd, Md = generate_rc_1d(D, fc, n_b, d_b, fyr, cov, dtu, is_frp=True, fcc=fcc, eccu=eccu)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = revolve_1d_pm_to_3d(Pn, Mn, Pd, Md)
            _, _, Pdb, Mdb = generate_rc_1d(D, fc, n_b, d_b, fyr, cov, dtu)
            M_db_full, P_db_full = np.concatenate([-Mdb[::-1], Mdb]), np.concatenate([Pdb[::-1], Pdb])
            _, _, _, P_surf_db, Mx_surf_db, My_surf_db = revolve_1d_pm_to_3d(Pn, Mn, Pdb, Mdb)
            draw_circ_section(ax_geom, D, n_b, d_b, cov, dtu, is_frp=True, n_ply=n_ply, t_ply=tf)
            Ec, fr, Ag = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), 0.25*math.pi*D**2
            Isr = sum([0.25*math.pi*d_b**2 * ((D/2-cov-dtu-d_b/2) * math.cos(i * 2 * math.pi / n_b))**2 for i in range(n_b)]) if n_b > 0 else 0
            EI_uncracked = (Ec * (math.pi*D**4/64) + ES_REBAR * Isr) / 1e9
            EI_cracked = (0.35 * Ec * (math.pi*D**4/64) + ES_REBAR * Isr) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "PERKUATAN FRP BUNDAR (ACI 440.2R-17)", f"D={D} mm", f"Awal: {fc} | Terkekang: {fcc:.1f} MPa", f"Po: {(0.85*fcc*(Ag-Asr)+fyr*Asr)/1000.0:,.1f} kN", f"Rebar: {n_b}-D{d_b}", f"FRP: {n_ply} Ply (fl {fl:.2f} MPa)"
            mcr_t, ei_u_t, ei_c_t, r1_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(D/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%"

        elif mode == '9. Kolom Persegi (RC) + FRP Wrap':
            B, H, fc, nc, nr, d_b, fyr, cov, stie, fytie, n_ply, tf, Ef, efu, CE, rc = w_B.value, w_H.value, w_fc.value, w_n_col.value, w_n_row.value, w_d_rebar.value, w_fy_rebar.value, w_cover.value, w_s_tie.value, w_fy_tie.value, w_frp_n.value, w_frp_tf.value, w_frp_Ef.value, w_frp_efu.value, w_frp_ce.value, w_frp_rc.value
            dtu = get_dtie(min(B-2*cov, H-2*cov), B*H, (B-2*cov)*(H-2*cov), fc, fytie, stie)
            r_coords = get_rect_rebar_coords(nc, nr, B-2*cov-2*dtu-d_b, H-2*cov-2*dtu-d_b)
            Asr = len(r_coords) * 0.25*math.pi*d_b**2
            fcc, eccu, fl = calc_frp_confinement(B, H, rc, fc, Asr, n_ply, tf, Ef, efu, CE, is_circ=False)
            X_c, Y_c, A_c = build_rect_conc_mesh(B, H)
            X_r, Y_r, A_r = np.array(r_coords)[:,0], np.array(r_coords)[:,1], 0.25*math.pi*d_b**2
            Pn, Mn, Pd, Md = generate_rc_rect_1d(B, H, fc, nc, nr, d_b, fyr, cov, dtu, is_frp=True, fcc=fcc, eccu=eccu)
            P_surf_n, Mx_surf_n, My_surf_n, P_surf_d, Mx_surf_d, My_surf_d = evaluate_3d_mesh(X_c, Y_c, A_c, X_r, Y_r, A_r, fyr, [], [], 0, 0, fc, True, fcc, eccu, get_beta1(fc), eccu, ES_REBAR, get_phi_tied)

            base_eval = evaluate_3d_mesh(X_c, Y_c, A_c, X_r, Y_r, A_r, fyr, [], [], 0, 0, fc, False, 0, 0, get_beta1(fc), CONC_CU, ES_REBAR, get_phi_tied)
            P_surf_db, Mx_surf_db, My_surf_db = base_eval[3], base_eval[4], base_eval[5]
            _, _, Pdb, Mdb = generate_rc_rect_1d(B, H, fc, nc, nr, d_b, fyr, cov, dtu)
            M_db_full, P_db_full = np.concatenate([-Mdb[::-1], Mdb]), np.concatenate([Pdb[::-1], Pdb])

            draw_rc_rect_section(ax_geom, B, H, nc, nr, d_b, cov, dtu, is_frp=True, n_ply=n_ply, t_ply=tf, rc=rc)
            Ec, fr, Ag = 4700*math.sqrt(fc), 0.62*math.sqrt(fc), B*H
            Isr = sum([A_r * y**2 for (_, y) in r_coords])
            EI_uncracked = (Ec * (B*H**3/12) + ES_REBAR * Isr) / 1e9
            EI_cracked = (0.35 * Ec * (B*H**3/12) + ES_REBAR * Isr) / 1e9
            tit_t, dim_t, fc_t, po_t, reb_t, stl_t = "PERKUATAN FRP PERSEGI (ACI 440.2R-17)", f"BxH: {B}x{H} mm", f"Awal: {fc} | Terkekang: {fcc:.1f} MPa", f"Po: {(0.85*fcc*(Ag-Asr)+fyr*Asr)/1000.0:,.1f} kN", f"Rebar: {len(X_r)}-D{d_b}", f"FRP: {n_ply} Ply (fl {fl:.2f} MPa)"
            mcr_t, ei_u_t, ei_c_t, r1_t, r2_t = f"Mcr: {fr*(EI_uncracked*1e9/Ec)/(H/2)/1e6:,.1f} kNm", f"EI unc: {EI_uncracked:,.0f} kNm\u00B2", f"EI cr: {EI_cracked:,.0f} kNm\u00B2", f"\u03C1 rebar: {(Asr/Ag)*100:.2f}%", f"Sudut r_c: {rc} mm"

        # --- RE-ASSEMBLE FULL 1D ARRAYS ---
        if len(M_n_full) == 0:
            M_n_full, P_n_full = np.concatenate([-Mn[::-1], Mn]), np.concatenate([Pn[::-1], Pn])
            M_d_full, P_d_full = np.concatenate([-Md[::-1], Md]), np.concatenate([Pd[::-1], Pd])
            if mode in ['6. Spun Pile + Infill Beton', '8. Kolom Bulat (RC) + FRP Wrap', '9. Kolom Persegi (RC) + FRP Wrap']:
                if not (mode == '6. Spun Pile + Infill Beton' and w_n_isian.value == 0):
                    M_db_full, P_db_full = np.concatenate([-Mdb[::-1], Mdb]), np.concatenate([Pdb[::-1], Pdb])

      # --- D/C RATIO CALCULATION & DISPLAY ---
        dc_t, dc_val_str, status_text, dc_color = "", "N/A", "", "#1a1a1a"

        if len(mx_vals) > 0:
            max_dc = 0
            thetas_eval = np.linspace(0, 2*np.pi, P_surf_d.shape[1]-1, endpoint=False)
            for pu, mux, muy in zip(pu_vals, mx_vals, my_vals):
                mu_tot = math.hypot(mux, muy)
                if mu_tot == 0 and pu == 0: continue
                theta_u = math.atan2(muy, mux)
                if theta_u < 0: theta_u += 2*math.pi
                j_idx = np.argmin(np.abs(thetas_eval - theta_u))

                # Interpolasi M_cap pada sudut theta_u dan beban aksial pu
                P_slice = P_surf_d[:, j_idx]
                M_slice = np.sqrt(Mx_surf_d[:, j_idx]**2 + My_surf_d[:, j_idx]**2)
                sort_idx = np.argsort(P_slice)
                P_sorted, M_sorted = P_slice[sort_idx], M_slice[sort_idx]

                if pu > np.max(P_sorted) or pu < np.min(P_sorted):
                    dc = 9.99
                else:
                    M_cap = np.interp(pu, P_sorted, M_sorted)
                    dc = mu_tot / M_cap if M_cap > 0 else 9.99
                max_dc = max(max_dc, dc)

            dc_t = "ACTIVE"
            dc_color = '#2E7D32' if max_dc <= 1.0 else '#C62828'
            status_text = "SAFE" if max_dc <= 1.0 else "UNSAFE"
            dc_val_str = "> 2.0" if max_dc >= 9.99 else f"{max_dc:.2f}"
            if max_dc >= 9.99: status_text = "AXIAL FAIL"

        # --- D/C RATIO TEXT (Professional & Clean) ---
        if dc_t:
            ax_info.text(0.85, 0.70, "MAX D/C RATIO", fontsize=10, color=dc_color, ha='center')
            ax_info.text(0.85, 0.50, dc_val_str, fontsize=22, color=dc_color, ha='center')
            ax_info.text(0.85, 0.35, status_text, fontsize=10, color=dc_color, ha='center')

        # --- PRINT INFO BOARD ---
        ax_info.text(0.04, 0.75, tit_t, fontsize=13, fontweight='bold', color='#0D47A1')
        ax_info.plot([0.04, 0.75], [0.65, 0.65], color='#0D47A1', lw=1.5)
        ax_info.text(0.04, 0.40, dim_t, fontsize=11)
        ax_info.text(0.30, 0.40, fc_t, fontsize=11)
        ax_info.text(0.55, 0.40, po_t, fontsize=11, fontweight='bold', color='#B71C1C')

        ax_info.text(0.04, 0.20, reb_t, fontsize=11)
        ax_info.text(0.30, 0.20, stl_t if stl_t else r1_t, fontsize=11)
        ax_info.text(0.55, 0.20, r2_t if r2_t else r1_t, fontsize=11)

        if mcr_t: ax_info.text(0.04, -0.05, mcr_t, fontsize=11, fontweight='bold', color='#1565C0')
        if ei_u_t: ax_info.text(0.30, -0.05, ei_u_t, fontsize=11)
        if ei_c_t: ax_info.text(0.55, -0.05, ei_c_t, fontsize=11)
        if r3_t: ax_info.text(0.80, -0.05, r3_t, fontsize=11)

        # --- PLOT 2D P-M DIAGRAM ---
        ax_pm.plot(M_n_full, P_n_full, '--', color='#1976D2', lw=2.0, label='Kapasitas Nominal (Pn)')
        if mode in ['6. Spun Pile + Infill Beton', '8. Kolom Bulat (RC) + FRP Wrap', '9. Kolom Persegi (RC) + FRP Wrap'] and not (mode == '6. Spun Pile + Infill Beton' and w_n_isian.value == 0):
            ax_pm.plot(M_db_full, P_db_full, '-.', color='#9E9E9E', lw=1.5, label='Desain Awal (Base)')
        ax_pm.plot(M_d_full, P_d_full, '-', color='#D32F2F', lw=2.5, label=r'Kapasitas Desain ($\phi$Pn)')

        if len(mx_vals) > 0:
            M_rs = [math.sqrt(x**2 + y**2) for x, y in zip(mx_vals, my_vals)]
            ax_pm.scatter(M_rs, pu_vals, color='#FF8F00', marker='o', s=40, label='Resultan Beban (Mu, Pu)', zorder=5)

        ax_pm.spines['left'].set_position('zero'); ax_pm.spines['bottom'].set_position('zero')
        ax_pm.spines['top'].set_visible(False); ax_pm.spines['right'].set_visible(False)
        ax_pm.set_title("DIAGRAM INTERAKSI P-M", fontsize=12, fontweight='bold', pad=20)
        ax_pm.set_xlabel("Momen (kNm)", loc='right'); ax_pm.set_ylabel("Aksial (kN)", loc='top', rotation=0, labelpad=-50)
        ax_pm.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='#e0e0e0')
        ax_pm.xaxis.set_major_locator(MaxNLocator(nbins=6)); ax_pm.yaxis.set_major_locator(MaxNLocator(nbins=8))
        ax_pm.grid(color='#ECEFF1', linestyle='-', linewidth=1.0)

        # --- PLOT 3D SURFACE ---
        ax_3d.grid(False)
        ax_3d.xaxis.pane.fill = ax_3d.yaxis.pane.fill = ax_3d.zaxis.pane.fill = False
        ax_3d.xaxis.pane.set_edgecolor('w'); ax_3d.yaxis.pane.set_edgecolor('w'); ax_3d.zaxis.pane.set_edgecolor('w')

        ax_3d.plot_wireframe(Mx_surf_n, My_surf_n, P_surf_n, color='#1976D2', linewidth=0.7, alpha=0.6)
        if mode in ['6. Spun Pile + Infill Beton', '8. Kolom Bulat (RC) + FRP Wrap', '9. Kolom Persegi (RC) + FRP Wrap'] and not (mode == '6. Spun Pile + Infill Beton' and w_n_isian.value == 0):
            ax_3d.plot_wireframe(Mx_surf_db, My_surf_db, P_surf_db, color='#9E9E9E', linewidth=0.5, alpha=0.5, linestyle='-.')

        ax_3d.plot_surface(Mx_surf_d, My_surf_d, P_surf_d, color='#D32F2F', alpha=0.15, edgecolor='#D32F2F', linewidth=0.5)

        proxy_n = mlines.Line2D([], [], color='#1976D2', label='Kapasitas Nominal (Pn)')
        proxy_d = mlines.Line2D([], [], color='#D32F2F', label=r'Kapasitas Desain ($\phi$Pn)')
        handles_3d = [proxy_n, proxy_d]

        if len(mx_vals) > 0:
            ax_3d.scatter(mx_vals, my_vals, pu_vals, color='#FF8F00', s=50, depthshade=False, alpha=1.0)
            handles_3d.append(mlines.Line2D([], [], color='#FF8F00', marker='o', linestyle='None', label='Beban (Mu_x, Mu_y, Pu)'))

        ax_3d.legend(handles=handles_3d, loc='upper right', frameon=True, facecolor='white', edgecolor='#e0e0e0')
        ax_3d.set_title("3D P-Mx-My Interaction Surface", fontsize=12, fontweight='bold', pad=20)
        ax_3d.set_xlabel('Momen X (kNm)', labelpad=10); ax_3d.set_ylabel('Momen Y (kNm)', labelpad=10); ax_3d.set_zlabel('Aksial P (kN)', labelpad=10)
        ax_3d.xaxis.set_major_locator(MaxNLocator(nbins=5)); ax_3d.yaxis.set_major_locator(MaxNLocator(nbins=5)); ax_3d.zaxis.set_major_locator(MaxNLocator(nbins=6))

        plt.tight_layout(pad=2.0)
        plt.show()

# ==============================================================================
# --- 6. INITIALIZE ---
# ==============================================================================
update_ui_visibility()
display(analisa_dropdown, ui_panel, output_area)

Dropdown(description='Mode Analisis:', layout=Layout(margin='0px 0px 10px 0px', width='320px'), options=('1. B…

Output(layout=Layout(overflow='hidden'))